# 10 機器學習 — 參考解答

松柏護理之家退伍軍人症群聚事件機器學習練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.inspection import permutation_importance

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

# 特徵定義
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]
feature_cols = num_cols + cat_cols + bin_cols

X = df[feature_cols]
y_infected = df["infected"]
y_severe = df["severe_outcome"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

## 題目 1：class_weight="balanced" 的效果

In [ ]:
# 無 class_weight
clf_default = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
scores_default = cross_val_score(clf_default, X, y_severe, cv=5, scoring="roc_auc")

# 有 class_weight="balanced"
clf_balanced = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=500, random_state=42,
                                 class_weight="balanced")),
])
scores_balanced = cross_val_score(clf_balanced, X, y_severe, cv=5, scoring="roc_auc")

print("=== Task B (severe_outcome) ===")
print(f"class_weight=None:       AUC = {scores_default.mean():.3f} \u00b1 {scores_default.std():.3f}")
print(f"class_weight='balanced': AUC = {scores_balanced.mean():.3f} \u00b1 {scores_balanced.std():.3f}")

print("\n\u2192 class_weight='balanced' 會對少數類別給予更高權重")
print("\u2192 在 AUC 上差異通常不大，但對 recall 有幫助")
print("\u2192 當正例比例很低（如 <10%）時，balanced 的效果更明顯")

## 題目 2：Task B 的特徵重要性

In [ ]:
# Random Forest on Task B
clf_rf = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores_rf_b = cross_val_score(clf_rf, X, y_severe, cv=5, scoring="roc_auc")
print(f"Random Forest 5-fold CV AUC (Task B) = {scores_rf_b.mean():.3f} \u00b1 {scores_rf_b.std():.3f}")

# Permutation importance
X_train, X_test, y_train, y_test = train_test_split(
    X, y_severe, test_size=0.3, random_state=42,
)
clf_rf.fit(X_train, y_train)

perm = permutation_importance(
    clf_rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc",
)

imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"],
        color="#e34a33", alpha=0.8)
ax.set_xlabel("Permutation Importance (AUC decrease)")
ax.set_title("Task B (severe_outcome) \u2014 Feature Importance")
plt.tight_layout()
plt.show()

print("\n=== Top 5 重要特徵（Task B）===")
for _, row in imp_df.nlargest(5, "importance").iterrows():
    print(f"  {row['feature']:25s}  importance = {row['importance']:.4f}")

print("\n\u2192 預測重症的重要特徵可能與預測感染不同")
print("\u2192 暴露因子（shower_use）可能對感染重要，但共病對重症更重要")

## 題目 3（挑戰題）：三模型比較 + ROC 曲線

In [ ]:
# 三個模型
models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocess", preprocess),
        ("model", GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ]),
}

# 70/30 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_infected, test_size=0.3, random_state=42,
)

# 訓練 + AUC + ROC
fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#2c7fb8", "#e34a33", "#41b6c4"]

print("=== Task A Test AUC ===")
for (name, clf), color in zip(models.items(), colors):
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2)
    print(f"  {name:25s}  AUC = {auc:.3f}")

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves \u2014 Task A (infected)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("\n\u2192 在 280 筆資料上，三個模型的表現通常很接近")
print("\u2192 單次 70/30 split 的結果不穩定，交叉驗證更可靠")
print("\u2192 小樣本的結論要保守解讀，避免過度宣稱模型效能")

### 解讀

- **class_weight**：在不平衡資料中，`balanced` 可提高少數類別的 recall，但 AUC 影響有限
- **Task A vs Task B**：預測感染的關鍵特徵（如 `shower_use`）和預測重症的關鍵特徵（如共病）可能不同，反映不同的因果機制
- **模型選擇**：280 筆資料不足以展現複雜模型的優勢。簡單模型 + 正確的交叉驗證 > 複雜模型 + 不當評估
- **ML vs 迴歸**：ML 強調預測，迴歸強調解釋。兩者互補——如果特徵重要性排序與 adjusted OR 方向一致，結論更具說服力